# Õppetund 16 - Skaalautuvate agendide juurutamine Microsoft Foundry abil

Selles märkmikus ehitate **tootmiskõlbuliku klienditoe agendi** väljamõeldud ettevõttele **Contoso**. Erinevalt varasematest õppetundidest ei seisne siinne eesmärk agendi mõtlemistsüklis — oluline on kõik, mis on selle ümber pakitud ja muudab agendi ohutuks suures mahus töötamiseks:

1. **Tööriistade kutsumine** — tellimuste otsingud ja piletite loomine.
2. **RAG** — poliitikavastused teadmistebaasist.
3. **Mälu** — kliendi meenutamine korduvate vestluste ajal.
4. **Mudelite marsruutimine** — lihtsad päringud väikesele mudelile, keerukamad suurele mudelile.
5. **Vastuse vahemällu salvestamine** — korduvate küsimuste teenindamine ilma mudelit kutsumata.
6. **Inimese kinnitamine** — üle künnise tagasimaksed peatatakse heakskiidu ootamiseks.
7. **Hindamise värav** — offline testikomplekt, mis blokeerib halbade väljaannete leviku.
8. **Jälgitavus** — OpenTelemetry jälgimine iga päringu ümber.

Iga jaotis on iseseisev ja käivitatav. Loe iga rida — tootmispõhimõtted on teadlikult hoitud väikestena.


## Häälestus

Enne selle märkmiku käivitamist veenduge, et teil on:

1. **Microsoft Foundry projekt** koos juurutatud vestlusmudeliga (nt `gpt-4.1-mini`).
2. **Sisselogitud Azure CLI-ga** — käivitage oma terminalis käsk `az login`.
3. **Seadistatud nõutud keskkonnamuutuja(d):**
   - `AZURE_AI_PROJECT_ENDPOINT` — teie Microsoft Foundry projekti lõpp-punkt.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — teie juurutatud mudeli nimi.

RAG jaotis kasutab **Azure AI Searchi** kui on seatud `AZURE_SEARCH_SERVICE_ENDPOINT` ja `AZURE_SEARCH_API_KEY`, muidu kasutab märkmik tööks mälus otsingut ilma Search ressursita.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import re
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential(),
)
print("Foundry client ready.")

## 1. Tööriistad

Tootetööriistad teevad päristööd pärissüsteemide vastu. Siin me simuleerime tellimuste andmebaasi ja piletisüsteemi lihtsate Pythoni funktsioonidega. `@tool` dekoratsioon avaldab need agendi jaoks.

Pane tähele, et `issue_refund` kasutab `approval_mode="always_require"` tagasimaksete korral, mis ületavad künnise — see on inimeste kaasamise primitive, mille me hiljem kasutusele võtame.


In [ ]:
# Simulated backend systems (in production these are API calls behind scoped identities).
ORDERS = {
    "A1001": {"status": "shipped", "total": 42.00, "eta": "2 days"},
    "A1002": {"status": "processing", "total": 128.50, "eta": "5 days"},
    "A1003": {"status": "delivered", "total": 19.99, "eta": "delivered"},
}
TICKETS: list[dict] = []
REFUND_APPROVAL_THRESHOLD = 50.0


@tool(approval_mode="never_require")
def get_order_status(order_id: Annotated[str, "The customer's order ID, e.g. A1001"]) -> str:
    """Look up the status of a customer order."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return f"No order found with ID {order_id}."
    return (
        f"Order {order_id.upper()}: status={order['status']}, "
        f"total=${order['total']:.2f}, eta={order['eta']}."
    )


@tool(approval_mode="never_require")
def open_ticket(
    subject: Annotated[str, "Short subject line for the support ticket"],
    details: Annotated[str, "Full description of the customer's issue"],
) -> str:
    """Open a support ticket for issues that need human follow-up."""
    ticket_id = f"T{1000 + len(TICKETS) + 1}"
    TICKETS.append({"id": ticket_id, "subject": subject, "details": details})
    return f"Ticket {ticket_id} opened: {subject}"


def refund_needs_approval(amount: float) -> bool:
    """Refunds above the threshold require a human approver."""
    return amount > REFUND_APPROVAL_THRESHOLD


@tool(approval_mode="always_require")
def issue_refund(
    order_id: Annotated[str, "The order to refund"],
    amount: Annotated[float, "Refund amount in USD"],
) -> str:
    """Issue a refund. Execution pauses for human approval before it runs."""
    return f"Refund of ${amount:.2f} issued for order {order_id.upper()}."


print("Tools defined.")

## 2. RAG — Poliitikate Teadmusbaas

Poliitikaküsimustele („milline on teie tagastamistähtaeg?“) tuleks vastata autoriteetsest allikast, mitte mudeli mälust. Me mähime väikese teadmusbaasi otsinguvahendina.

Tootmiskeskkonnas on see **Azure AI Search**; siin pakume mälupõhist märksõnaotsingut, et sülearvuti töötaks kõikjal, ja lülitume automaatselt Azure AI Searchile, kui keskkonnamuutujad on olemas.


In [ ]:
KNOWLEDGE_BASE = {
    "returns": "Contoso accepts returns within 30 days of delivery for a full refund. Items must be unused and in original packaging.",
    "shipping": "Standard shipping takes 3-5 business days. Express shipping (1-2 days) is available at checkout for an extra fee.",
    "warranty": "All Contoso electronics carry a 12-month limited warranty covering manufacturing defects.",
    "refund_policy": "Refunds are processed to the original payment method within 5 business days of approval. Refunds over $50 require a supervisor's approval.",
}


def _in_memory_search(query: str) -> str:
    q = query.lower()
    hits = [text for key, text in KNOWLEDGE_BASE.items() if key.replace("_", " ") in q or key in q]
    if not hits:
        # crude keyword fallback so the tool still returns something useful
        hits = [text for text in KNOWLEDGE_BASE.values() if any(w in text.lower() for w in q.split())]
    return "\n".join(hits) if hits else "No matching policy found."


def _azure_search(query: str) -> str:
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient

    client = SearchClient(
        endpoint=os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"],
        index_name=os.getenv("AZURE_SEARCH_INDEX_NAME", "contoso-policies"),
        credential=AzureKeyCredential(os.environ["AZURE_SEARCH_API_KEY"]),
    )
    results = client.search(search_text=query, top=3)
    return "\n".join(r.get("content", "") for r in results) or "No matching policy found."


USE_AZURE_SEARCH = bool(os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT") and os.getenv("AZURE_SEARCH_API_KEY"))


@tool(approval_mode="never_require")
def search_policies(query: Annotated[str, "The policy question to look up"]) -> str:
    """Search Contoso support policies to answer customer questions."""
    if USE_AZURE_SEARCH:
        return _azure_search(query)
    return _in_memory_search(query)


print(f"RAG ready. Using {'Azure AI Search' if USE_AZURE_SEARCH else 'in-memory search'}.")

## 3. Mälu

Tugispetsialist, kes unustab, kellega ta räägib, on halb tugispetsialist. Me hoiame kliendipõhist pisikest profiilipoodi ja lisame lühikese kokkuvõtte agendi juhistesse. Tootmises on see mäluteenus (vt õppetund 13); siin teeb sõnastik mustri nähtavaks.


In [ ]:
CUSTOMER_MEMORY: dict[str, dict] = {
    "cust-42": {"name": "Dana", "tier": "enterprise", "recent_order": "A1002"},
    "cust-99": {"name": "Sam", "tier": "standard", "recent_order": "A1003"},
}


def memory_context(customer_id: str) -> str:
    profile = CUSTOMER_MEMORY.get(customer_id)
    if not profile:
        return "This is a new customer with no history."
    return (
        f"Customer {profile['name']} ({profile['tier']} tier). "
        f"Most recent order: {profile['recent_order']}."
    )


print(memory_context("cust-42"))

## 4 & 5. Mudelite marsruutimine ja vastuste vahemällu salvestamine

Kaks kululeverit on ühendatud ühe taotluskäitlejaga:

- **Marsruutimine**: odav heuristiline klassifikaator otsustab, kas taotlus vajab väikest või suurt mudelit.
- **Vahemällu salvestamine**: normaliseeritud korduvad küsimused teenindatakse otse vahemälust ilma mudeli kutseta.

Klassifikaator on siin tahtlikult lihtne. Tootmises valideeriksite seda liikluse vastu ja võiks selle asendada Foundry Model Routeriga.


In [ ]:
SMALL_MODEL = os.getenv("AZURE_AI_SMALL_MODEL", model)   # e.g. gpt-4.1-mini
LARGE_MODEL = os.getenv("AZURE_AI_LARGE_MODEL", model)   # e.g. gpt-4.1

response_cache: dict[str, str] = {}
route_counters = {"small": 0, "large": 0, "cache": 0}


def normalize(query: str) -> str:
    return re.sub(r"\s+", " ", query.lower().strip())


COMPLEX_SIGNALS = ("refund", "cancel", "complaint", "escalate", "broken", "wrong", "why")


def is_simple(query: str) -> bool:
    """Route complex or high-stakes requests to the large model; everything else to the small one."""
    q = query.lower()
    if any(signal in q for signal in COMPLEX_SIGNALS):
        return False
    return len(q.split()) <= 20


def choose_model(query: str) -> str:
    return SMALL_MODEL if is_simple(query) else LARGE_MODEL


print(f"Small model: {SMALL_MODEL} | Large model: {LARGE_MODEL}")

## 6 & 8. Agent, inimkinnitus ja jälgitavus

Nüüd paigaldame agendi ülaltoodud tööriistadest ja pakime iga päringu OpenTelemetry lõigusse. Funktsioon `handle_support_request` on tootmiskeskkonna päringute töötleja: vahemälu → marsruut → jälgimine → käivitamine → vahemälu.

Inimkinnitus toimub raamistikus: kuna `issue_refund` on `approval_mode="always_require"`, peatub käivitamine ja kuvab kinnitustaotluse, mida lahendame otseselt.


In [ ]:
# Tracing: use the Agent Framework tracer if available, else a no-op so the notebook runs anywhere.
try:
    from agent_framework.observability import get_tracer
    tracer = get_tracer()
except Exception:  # observability extras not installed
    from contextlib import contextmanager

    class _NoopSpan:
        def set_attribute(self, *_args, **_kwargs):
            pass

    class _NoopTracer:
        @contextmanager
        def start_as_current_span(self, _name):
            yield _NoopSpan()

    tracer = _NoopTracer()


SUPPORT_INSTRUCTIONS = (
    "You are Contoso's customer support agent. Be concise, friendly, and accurate. "
    "Use search_policies for policy questions, get_order_status for orders, "
    "open_ticket when a human needs to follow up, and issue_refund for refunds. "
    "Never invent policy details."
)

support_agent = provider.as_agent(
    name="ContosoSupportAgent",
    instructions=SUPPORT_INSTRUCTIONS,
    tools=[get_order_status, open_ticket, search_policies, issue_refund],
)


async def handle_support_request(query: str, customer_id: str) -> str:
    # 1. Serve from cache when we can.
    key = normalize(query)
    if key in response_cache:
        route_counters["cache"] += 1
        return response_cache[key]

    # 2. Route by complexity to control cost.
    chosen_model = choose_model(query)
    route_counters["small" if chosen_model == SMALL_MODEL else "large"] += 1

    # 3. Add per-customer memory to the prompt.
    context = memory_context(customer_id)
    prompt = f"[Customer context: {context}]\n\n{query}"

    # 4. Run inside a trace span for observability.
    with tracer.start_as_current_span("support_request") as span:
        span.set_attribute("customer.id", customer_id)
        span.set_attribute("routed.model", chosen_model)
        response = await support_agent.run(prompt, model=chosen_model)

    text = response.text
    response_cache[key] = text
    return text


print("Support agent assembled.")

In [ ]:
# Try a few requests. The first is simple (small model), the second is a refund (large model + approval path).
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print(await handle_support_request("Where is my order A1002?", "cust-42"))
print("---")
# Repeat the first question -> served from cache.
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print("Routing counters:", route_counters)

## 7. Hindamislukk

See on tunni avaldamislukk: agenti hindab võrguühenduseta testikomplekt ja juurutamine jätkub ainult siis, kui soorituse määr ületab künnise. Hindaja siin on lihtne märksõnade kattuvuse kontroll, et hoida märkmik iseseisvana; tootmises kasutaksite LLM-i kohtunikuna või raamistikuhindajat (vt õppetund 10).


In [ ]:
TEST_CASES = [
    {"input": "How long do I have to return an item?", "expected": ["30 days", "refund"]},
    {"input": "How fast is standard shipping?", "expected": ["3-5", "business days"]},
    {"input": "What is the status of order A1001?", "expected": ["shipped", "A1001"]},
    {"input": "Do your electronics have a warranty?", "expected": ["12-month", "warranty"]},
]


def score_response(actual: str, expected_keywords: list[str]) -> float:
    actual_l = actual.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in actual_l)
    return hits / len(expected_keywords)


async def evaluation_gate(test_cases: list[dict], threshold: float = 0.8) -> bool:
    passed = 0
    for case in test_cases:
        result = await support_agent.run(case["input"])
        s = score_response(result.text, case["expected"])
        status = "PASS" if s >= 0.5 else "FAIL"
        print(f"[{status}] {case['input']}  (score={s:.0%})")
        if s >= 0.5:
            passed += 1
    pass_rate = passed / len(test_cases)
    print(f"\nEvaluation pass rate: {pass_rate:.0%} (gate: {threshold:.0%})")
    return pass_rate >= threshold


gate_passed = await evaluation_gate(TEST_CASES, threshold=0.8)
print("\nDeploy allowed:" , gate_passed)

## Kokkupanek: simuleeritud väljalase

Allolev rakk näitab kogu silmust, mida õppetund kirjeldab: käivita hindamisvärav ja "võta kasutusele" ainult siis, kui see läbib. See on muster, mida käitataks CI-s enne agentversiooni edendamist Foundry Agent Service'i.


In [ ]:
async def release(test_cases: list[dict]) -> None:
    print("Running pre-deployment evaluation gate...\n")
    if await evaluation_gate(test_cases, threshold=0.8):
        print("\n✅ Gate passed — promoting agent version to the Foundry Agent Service.")
    else:
        print("\n❌ Gate failed — release blocked. Fix the agent and re-run.")


await release(TEST_CASES)

## Kokkuvõte

Sa koostasid tootmiseks valmis klienditoe agendi, kus iga operatiivne aspekt on ühendatud:

- **Tööriistad, RAG ja mälu** annavad agentidele võimekuse ja konteksti.
- **Mudelivalik ja vahemällu salvestamine** hoiavad latentsuse ja kulud kontrolli all.
- **Inimese heakskiit** kaitseb kõrge riskiga toimingute, nagu suure tagasimakse, vastu.
- **Hindamise värav** blokeerib halvad väljalasked enne nende saatmist.
- **Jälgimine** teeb iga päringu jälgitavaks.

### Väljakutse

Laienda seda agenti, et:

1. **Toetada mitut mudelit** — lisa kolmas "põhjendamise" tase ja suuna eskalatsioonid/kaebused sinna.
2. **Lisa hindamisväravad** — laienda `TEST_CASES` refundikinnituse stsenaariumitega ja kinnita, et värav püüab regressioone kinni.
3. **Lisa kulu-teadlik marsruutimine** — jälgi hinnangulist kulu päringu kohta (väike vs suur vs vahemällu) ja prindi kulude aruanne pärast segapaketi päringuid.

Järgmises õppetükis teed vastupidise teekonna ja jooksutad agenti täielikult oma masinas Microsoft Foundry Locali ja Qweni abil.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Lahtiütlus**:
See dokument on tõlgitud kasutades AI tõlketeenust [Co-op Translator](https://github.com/Azure/co-op-translator). Kuigi me püüdleme täpsuse poole, palun pange tähele, et automatiseeritud tõlgetes võib esineda vigu või ebatäpsusi. Originaaldokument selle emakeeles tuleks pidada autoriteetseks allikaks. Olulise teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta selle tõlkega seotud eksimustest või valesti mõistmistest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
